# 03 Arrival-Only Modeling
Baseline and residual models using only arrival-time available features.

Prerequisite: run 01_data_prep.ipynb first.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data_dir = Path('../data')
model_A_df = pd.read_csv(data_dir / 'model_A_arrival_dataset.csv')

for c in ['WAITTIME', 'YEAR', 'ARRIVAL_HOUR', 'VMONTH', 'VDAYR', 'AGE', 'PAINSCALE', 'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT', 'IMMEDR', 'SEX', 'RACEUN', 'ETHUN']:
    if c in model_A_df.columns:
        model_A_df[c] = pd.to_numeric(model_A_df[c], errors='coerce')

sentinel_by_col = {
    'PAINSCALE': {-9, 99},
    'BPSYS': {-9, 998, 999},
    'BPDIAS': {-9, 998, 999},
    'TEMPF': {-9, 998, 999, 9980, 9990, 9989, 9999},
    'PULSE': {-9, 998, 999},
    'RESPR': {-9, 98, 99, 998, 999},
    'POPCT': {-9, 998, 999},
    'IMMEDR': {-8, -9, 8, 9, 98, 99},
    'SEX': {-9, 9, 99},
    'RACEUN': {-9, 9, 99},
    'ETHUN': {-9, 9, 99}
}

for col, bad_vals in sentinel_by_col.items():
    if col in model_A_df.columns:
        model_A_df.loc[model_A_df[col].isin(bad_vals), col] = np.nan

if 'TEMPF' in model_A_df.columns:
    temp = model_A_df['TEMPF'].copy()
    temp = np.where((temp >= 850) & (temp <= 1150), temp / 10.0, temp)
    model_A_df['TEMPF'] = pd.to_numeric(temp, errors='coerce')

model_A_df = model_A_df[model_A_df['WAITTIME'].between(0, 480)].copy()
model_A_df.to_csv(data_dir / 'model_A_arrival_dataset_cleaned.csv', index=False)
print('Cleaned shape:', model_A_df.shape)

Cleaned shape: (91812, 18)


In [2]:
split_years = {'train': [2015, 2016, 2017, 2018], 'valid': [2021], 'holdout': [2022]}
train_df = model_A_df[model_A_df['YEAR'].isin(split_years['train'])].copy()
valid_df = model_A_df[model_A_df['YEAR'].isin(split_years['valid'])].copy()
holdout_df = model_A_df[model_A_df['YEAR'].isin(split_years['holdout'])].copy()

print('train:', len(train_df), 'valid:', len(valid_df), 'holdout:', len(holdout_df))

train_hour_median = train_df.groupby('ARRIVAL_HOUR', dropna=False)['WAITTIME'].median()
train_global_median = train_df['WAITTIME'].median()

def baseline_predict(frame):
    pred = frame['ARRIVAL_HOUR'].map(train_hour_median)
    return pd.to_numeric(pred, errors='coerce').fillna(train_global_median)

train: 64766 valid: 13830 holdout: 13216


In [3]:
feature_cols = [
    'ARRIVAL_HOUR', 'VMONTH', 'VDAYR', 'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR',
    'PAINSCALE', 'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT'
]
feature_cols = [c for c in feature_cols if c in model_A_df.columns]
cat_cols = [c for c in ['SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'VDAYR', 'VMONTH'] if c in feature_cols]
num_cols = [c for c in feature_cols if c not in cat_cols]

def prep_features(frame, fit_maps=None):
    out = frame.copy()
    for c in num_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    if 'ARRIVAL_HOUR' in out.columns:
        h = pd.to_numeric(out['ARRIVAL_HOUR'], errors='coerce')
        out['hour_sin'] = np.sin(2 * np.pi * h / 24.0)
        out['hour_cos'] = np.cos(2 * np.pi * h / 24.0)
    if 'VMONTH' in out.columns:
        m = pd.to_numeric(out['VMONTH'], errors='coerce')
        out['month_sin'] = np.sin(2 * np.pi * m / 12.0)
        out['month_cos'] = np.cos(2 * np.pi * m / 12.0)

    maps = {} if fit_maps is None else fit_maps
    for c in cat_cols:
        s = out[c].astype('string').fillna('missing')
        if fit_maps is None:
            maps[c] = {v: i for i, v in enumerate(pd.Index(s.dropna().unique()))}
        out[c] = s.map(maps[c]).fillna(-1).astype(float)

    cols = num_cols + cat_cols + [c for c in ['hour_sin', 'hour_cos', 'month_sin', 'month_cos'] if c in out.columns]
    X = out[cols].apply(pd.to_numeric, errors='coerce')
    X = X.fillna(X.median(numeric_only=True))
    return X, maps

X_train, fmap = prep_features(train_df, fit_maps=None)
X_valid, _ = prep_features(valid_df, fit_maps=fmap)
X_holdout, _ = prep_features(holdout_df, fit_maps=fmap)

y_train = pd.to_numeric(train_df['WAITTIME'], errors='coerce')
y_valid = pd.to_numeric(valid_df['WAITTIME'], errors='coerce')
y_holdout = pd.to_numeric(holdout_df['WAITTIME'], errors='coerce')

base_train = baseline_predict(train_df)
base_valid = baseline_predict(valid_df)
base_holdout = baseline_predict(holdout_df)

resid_model = HistGradientBoostingRegressor(
    loss='absolute_error',
    learning_rate=0.05,
    max_iter=350,
    max_depth=8,
    min_samples_leaf=40,
    l2_regularization=0.1,
    random_state=42
)
resid_model.fit(X_train, y_train - base_train)

pred_valid = np.clip(base_valid + resid_model.predict(X_valid), 0, 480)
pred_holdout = np.clip(base_holdout + resid_model.predict(X_holdout), 0, 480)

eval_df = pd.DataFrame([
    {'model_name': 'by_hour_median_trainfit', 'split': 'valid', 'mae': mean_absolute_error(y_valid, base_valid), 'rmse': np.sqrt(mean_squared_error(y_valid, base_valid)), 'r2': r2_score(y_valid, base_valid)},
    {'model_name': 'residual_hgb', 'split': 'valid', 'mae': mean_absolute_error(y_valid, pred_valid), 'rmse': np.sqrt(mean_squared_error(y_valid, pred_valid)), 'r2': r2_score(y_valid, pred_valid)},
    {'model_name': 'by_hour_median_trainfit', 'split': 'holdout', 'mae': mean_absolute_error(y_holdout, base_holdout), 'rmse': np.sqrt(mean_squared_error(y_holdout, base_holdout)), 'r2': r2_score(y_holdout, base_holdout)},
    {'model_name': 'residual_hgb', 'split': 'holdout', 'mae': mean_absolute_error(y_holdout, pred_holdout), 'rmse': np.sqrt(mean_squared_error(y_holdout, pred_holdout)), 'r2': r2_score(y_holdout, pred_holdout)}
])
display(eval_df)
eval_df.to_csv(data_dir / 'model_eval_summary_improved.csv', index=False)

,model_name,split,mae,rmse,r2
0,by_hour_median_trainfit,valid,25.692697,50.744881,-0.086506
1,residual_hgb,valid,25.551924,50.496751,-0.075906
2,by_hour_median_trainfit,holdout,28.344885,55.999156,-0.100491
3,residual_hgb,holdout,28.364057,55.902335,-0.096689


In [4]:
high_acuity_codes = {1.0, 2.0}

def assign_segment(frame):
    if 'IMMEDR' not in frame.columns:
        return pd.Series(['other'] * len(frame), index=frame.index)
    return np.where(frame['IMMEDR'].isin(high_acuity_codes), 'high_acuity', 'other')

segment_models = {}
segment_maps = {}
for segment_name in ['high_acuity', 'other']:
    tr = train_df[assign_segment(train_df) == segment_name].copy()
    if len(tr) < 200:
        continue
    X_tr, fmap_seg = prep_features(tr, fit_maps=None)
    y_tr = pd.to_numeric(tr['WAITTIME'], errors='coerce')
    resid_tr = y_tr - baseline_predict(tr)
    m = HistGradientBoostingRegressor(loss='absolute_error', learning_rate=0.04, max_iter=350, max_depth=7, min_samples_leaf=30, l2_regularization=0.1, random_state=42)
    m.fit(X_tr, resid_tr)
    segment_models[segment_name] = m
    segment_maps[segment_name] = fmap_seg

def predict_segmented(frame):
    f = frame.copy()
    seg = assign_segment(f)
    base = baseline_predict(f)
    pred = base.copy()
    for segment_name in ['high_acuity', 'other']:
        idx = f.index[seg == segment_name]
        if len(idx) == 0 or segment_name not in segment_models:
            continue
        sub = f.loc[idx]
        X_sub, _ = prep_features(sub, fit_maps=segment_maps[segment_name])
        pred.loc[idx] = np.clip(base.loc[idx] + segment_models[segment_name].predict(X_sub), 0, 480)
    return pred

pred_holdout_seg = predict_segmented(holdout_df)
seg_eval = pd.DataFrame([
    {'model_name': 'by_hour_median_trainfit', 'split': 'holdout', 'mae': mean_absolute_error(y_holdout, base_holdout), 'rmse': np.sqrt(mean_squared_error(y_holdout, base_holdout)), 'r2': r2_score(y_holdout, base_holdout)},
    {'model_name': 'segmented_hgb_by_immedr', 'split': 'holdout', 'mae': mean_absolute_error(y_holdout, pred_holdout_seg), 'rmse': np.sqrt(mean_squared_error(y_holdout, pred_holdout_seg)), 'r2': r2_score(y_holdout, pred_holdout_seg)}
])
display(seg_eval)

holdout_cmp = holdout_df.copy()
holdout_cmp['segment'] = assign_segment(holdout_cmp)
holdout_cmp['y_true'] = y_holdout
holdout_cmp['pred_base'] = base_holdout
holdout_cmp['pred_segmented'] = pred_holdout_seg
holdout_cmp['ae_base'] = (holdout_cmp['y_true'] - holdout_cmp['pred_base']).abs()
holdout_cmp['ae_segmented'] = (holdout_cmp['y_true'] - holdout_cmp['pred_segmented']).abs()
holdout_cmp['mae_gain'] = holdout_cmp['ae_base'] - holdout_cmp['ae_segmented']

slice_report = holdout_cmp.groupby('segment', dropna=False).agg(rows=('y_true', 'size'), mae_base=('ae_base', 'mean'), mae_segmented=('ae_segmented', 'mean'), mae_gain=('mae_gain', 'mean')).reset_index()
display(slice_report)

seg_eval.to_csv(data_dir / 'model_eval_summary_segmented.csv', index=False)
holdout_cmp.to_csv(data_dir / 'holdout_predictions_segmented.csv', index=False)
slice_report.to_csv(data_dir / 'holdout_slice_report_segmented.csv', index=False)

,model_name,split,mae,rmse,r2
0,by_hour_median_trainfit,holdout,28.344885,55.999156,-0.100491
1,segmented_hgb_by_immedr,holdout,28.372584,55.844995,-0.094440


,segment,rows,mae_base,mae_segmented,mae_gain
0,high_acuity,1461,22.416838,22.302306,0.114531
1,other,11755,29.081667,29.127044,-0.045376
